# Assessment of Soil Microplastic Contamination: Distribution, Soil Properties, and Agricultural Practices

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# PROJECT DIRECTORIES

PROJECT_DIR = os.getcwd()

DATA_DIR = os.path.join(PROJECT_DIR, "raw data")
FIG_DIR = "figures"
TABLE_DIR = "tables"
OUTPUT_DIR = "cleaned_data"

# CREATE OUTPUT FOLDERS

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# VISUALIZATION SETTINGS

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

print("Project setup completed.")
print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)

In [ ]:
FILES = {
    "mp_quant": "surveympquantification.csv",
    "soil_props": "surveysoilproperties.csv",
    "questionnaire": "surveyquestionnaire.csv",
    "mp_id": "surveympidentification.csv",
    "std_quant": "standardmpquantification.csv",
    "std_polymer": "standardpolymerid.csv",
    "std_lib_uk": "standardpolymerlibrarycomparisonuk.csv",
    "lab_meta": "laboratorymetadata.csv"
}

print("Total datasets:", len(FILES))

for name, filename in FILES.items():
    print(name, ":", filename)

# 3. LOAD ALL DATASETS

In [ ]:
data = {}

for name, filename in FILES.items():

    path = os.path.join(DATA_DIR, filename)

    if os.path.exists(path):

        try:
            data[name] = pd.read_csv(
                path,
                encoding="latin1"
            )

            print(
                f"[OK] {name}: "
                f"{data[name].shape[0]} rows x "
                f"{data[name].shape[1]} columns"
            )

        except Exception as e:
            data[name] = None
            print(f"[ERROR] {filename}: {e}")

    else:
        data[name] = None
        print(f"[MISSING] {filename}")


# CHECK REQUIRED DATASETS

required_datasets = [
    "mp_quant",
    "soil_props",
    "questionnaire"
]

for name in required_datasets:

    if data[name] is None:
        raise FileNotFoundError(
            f"Required dataset is missing: {FILES[name]}"
        )


# LOADING SUMMARY

loaded_count = sum(
    df is not None for df in data.values()
)

print("\nDATASET LOADING SUMMARY")
print("-----------------------")
print("Total datasets:", len(FILES))
print("Successfully loaded:", loaded_count)
print("Not loaded:", len(FILES) - loaded_count)

print("\nRequired datasets are available.")

#4. INITIAL DATASET INSPECTION

In [ ]:
# INITIAL DATASET INSPECTION

for name, df in data.items():

    if df is None:
        continue

    print("\n" + "=" * 60)
    print("DATASET:", name)
    print("Shape:", df.shape)
    print("=" * 60)

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nFirst 3 rows:")
    print(df.head(3))

# 5. INITIAL DATA QUALITY REPORT

In [ ]:
# INITIAL DATA QUALITY REPORT

quality_reports = []

for name, df in data.items():

    if df is None:
        continue

    report = pd.DataFrame({
        "Dataset": name,
        "Column": df.columns,
        "Data_Type": [str(df[col].dtype) for col in df.columns],
        "Missing_Values": [df[col].isna().sum() for col in df.columns],
        "Missing_Percentage": [
            round(df[col].isna().mean() * 100, 2)
            for col in df.columns
        ],
        "Unique_Values": [df[col].nunique() for col in df.columns]
    })

    quality_reports.append(report)

    print(f"\nDataset: {name}")
    print("Duplicate rows:", df.duplicated().sum())


# COMBINE ALL QUALITY REPORTS

quality_report = pd.concat(
    quality_reports,
    ignore_index=True
)

# DISPLAY QUALITY REPORT

quality_report = quality_report.sort_values(
    by=["Dataset", "Missing_Percentage"],
    ascending=[True, False]
)

quality_report

# 6. save quality report

In [ ]:
quality_report.to_csv(
    os.path.join(TABLE_DIR, "initial_data_quality_report.csv"),
    index=False
)

print("Initial data quality report saved successfully.")

# 7. CLEANING FUNCTIONS

In [ ]:
# MISSING VALUE REPRESENTATIONS

missing_values = [
    "",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "None",
    "none"
]


# COLUMN NAME STANDARDIZATION

column_aliases = {
    "Site Code": "Site",
    "Site_Code": "Site"
}


# FUNCTION TO CLEAN A DATAFRAME

def clean_dataframe(df):

    cleaned_df = df.copy()

    # Remove extra spaces from column names
    cleaned_df.columns = cleaned_df.columns.str.strip()

    # Standardize inconsistent column names
    cleaned_df = cleaned_df.rename(
        columns=column_aliases
    )

    # Remove extra spaces from text values
    for col in cleaned_df.select_dtypes(include="object").columns:

        cleaned_df[col] = cleaned_df[col].str.strip()

    # Standardize missing values
    cleaned_df = cleaned_df.replace(
        missing_values,
        np.nan
    )

    return cleaned_df


print("Cleaning functions defined successfully.")

# 8. test cleaning function

In [ ]:
# CLEAN ALL DATASETS

cleaned_data = {}

for name, df in data.items():

    if df is not None:

        cleaned_data[name] = clean_dataframe(df)

        print(
            f"{name}: "
            f"{cleaned_data[name].shape[0]} rows x "
            f"{cleaned_data[name].shape[1]} columns"
        )

print("\nAll available datasets cleaned successfully.")

In [ ]:
# MAIN CLEANED DATASETS

mp_quant_clean = cleaned_data["mp_quant"]
soil_props_clean = cleaned_data["soil_props"]
questionnaire_clean = cleaned_data["questionnaire"]

mp_id_clean = cleaned_data["mp_id"]
std_quant_clean = cleaned_data["std_quant"]
std_polymer_clean = cleaned_data["std_polymer"]
std_lib_uk_clean = cleaned_data["std_lib_uk"]
lab_meta_clean = cleaned_data["lab_meta"]

9: Convert numeric columns correctly

# 9. Clean the main survey datasets

In [ ]:
# NUMERIC COLUMNS FOR EACH DATASET

numeric_columns = {

    "mp_quant": [
        "Depth", "Z", "MP_number"
    ],

    "soil_props": [
        "Depth", "pH", "EC", "NH4",
        "NO3", "DOC", "PO4"
    ],

    "questionnaire": [
        "Max_cultivation_depth",
        "Annual_average_rainfall",
        "Annual_average_air_temperature",
        "PMF_use_duration",
        "Coated_fertiliser_use_duration",
        "Compost_quantity_per_year",
        "Compost_use_duration",
        "Sewage_sludge_quantity_per_year",
        "Sewage_sludge_use_duration"
    ],

    "mp_id": [
        "Depth", "Quality"
    ],

    "std_quant": [
        "Rep", "Particle_number"
    ],

    "std_polymer": [
        "Sample", "Particle", "MatchQuality"
    ],

    "std_lib_uk": [
        "Library", "Sample", "Particle", "MatchQuality"
    ],

    "lab_meta": [
        "Membrane_area", "Membrane_d",
        "Real_membrane_d", "Real_membrane_a",
        "Pore_size", "Emission_wavelength",
        "Nile_red_purity"
    ]
}


# CONVERT NUMERIC COLUMNS

numeric_conversion_report = []

for name, columns in numeric_columns.items():

    df = cleaned_data[name]

    for col in columns:

        if col in df.columns:

            original_non_missing = df[col].notna()

            converted = pd.to_numeric(
                df[col],
                errors="coerce"
            )

            failed_conversions = (
                original_non_missing & converted.isna()
            ).sum()

            numeric_conversion_report.append({
                "Dataset": name,
                "Column": col,
                "Failed_Conversions": failed_conversions
            })

            df[col] = converted


numeric_conversion_report = pd.DataFrame(
    numeric_conversion_report
)

numeric_conversion_report

# 10. Compare missing values before and after cleaning

In [ ]:
# COMPARE ORIGINAL AND CLEANED DATASETS

cleaning_summary = []

for name in cleaned_data:

    original_df = data[name]
    cleaned_df = cleaned_data[name]

    cleaning_summary.append({

        "Dataset": name,

        "Original_Rows": original_df.shape[0],
        "Cleaned_Rows": cleaned_df.shape[0],

        "Original_Columns": original_df.shape[1],
        "Cleaned_Columns": cleaned_df.shape[1],

        "Original_Missing": original_df.isna().sum().sum(),
        "Cleaned_Missing": cleaned_df.isna().sum().sum(),

        "Original_Duplicates": original_df.duplicated().sum(),
        "Cleaned_Duplicates": cleaned_df.duplicated().sum()

    })


cleaning_summary = pd.DataFrame(cleaning_summary)

cleaning_summary

# 11. Save the cleaning summary

In [ ]:
cleaning_summary.to_csv(
    os.path.join(TABLE_DIR, "cleaning_summary.csv"),
    index=False
)

numeric_conversion_report.to_csv(
    os.path.join(TABLE_DIR, "numeric_conversion_report.csv"),
    index=False
)

print("Cleaning reports saved successfully.")

# 12: Missing value analysis for all datasets

In [ ]:
# MISSING VALUE ANALYSIS

missing_report = []

for name, df in cleaned_data.items():

    for col in df.columns:

        missing_count = df[col].isna().sum()

        missing_report.append({
            "Dataset": name,
            "Column": col,
            "Missing_Count": missing_count,
            "Missing_Percentage": round(
                df[col].isna().mean() * 100, 2
            )
        })


missing_report = pd.DataFrame(missing_report)

# SHOW ONLY COLUMNS WITH MISSING VALUES

missing_report = missing_report[
    missing_report["Missing_Count"] > 0
]

missing_report = missing_report.sort_values(
    by=["Dataset", "Missing_Percentage"],
    ascending=[True, False]
)

missing_report

# 13: Save the missing value report

In [ ]:
missing_report.to_csv(
    os.path.join(TABLE_DIR, "missing_value_report.csv"),
    index=False
)

print("Missing value report saved successfully.")

# 14: Visualize missing values

In [ ]:
# MISSING VALUE VISUALIZATION

for name, df in cleaned_data.items():

    missing_counts = df.isna().sum()

    missing_counts = missing_counts[
        missing_counts > 0
    ]

    if missing_counts.empty:
        continue

    # Sort so the worst columns are most visible
    missing_counts = missing_counts.sort_values(ascending=False)

    # Red -> orange gradient: more missing = more intense
    bar_colors = sns.color_palette(
        "YlOrRd",
        n_colors=len(missing_counts)
    )

    plt.figure(figsize=(10, 5))

    sns.barplot(
        x=missing_counts.index,
        y=missing_counts.values,
        hue=missing_counts.index,
        palette=bar_colors,
        legend=False,
        edgecolor="black",
        linewidth=0.5
    )

    # Annotate values on top of each bar
    for i, v in enumerate(missing_counts.values):
        plt.text(
            i, v,
            str(v),
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold"
        )

    plt.title(f"Missing Values: {name}", fontweight="bold")
    plt.xlabel("Columns")
    plt.ylabel("Missing Values")
    plt.xticks(rotation=45, ha="right")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            FIG_DIR,
            f"{name}_missing_values.png"
        ),
        bbox_inches="tight"
    )

    plt.show()

# 15: Investigate questionnaire missing values

In [ ]:
# QUESTIONNAIRE MISSING VALUE INVESTIGATION

conditional_fields = {
    "Coated_fertiliser_use": [
        "Coated_fertiliser_use_duration"
    ],

    "Compost_use": [
        "Compost_quantity_per_year",
        "Compost_use_duration"
    ],

    "Sewage_sludge_use": [
        "Sewage_sludge_quantity_per_year",
        "Sewage_sludge_use_duration"
    ]
}

for question, related_fields in conditional_fields.items():

    print("\nQuestion:", question)

    print("\nResponse distribution:")
    print(questionnaire_clean[question].value_counts(dropna=False))

    for field in related_fields:

        print(f"\nMissing {field} by {question}:")

        print(
            questionnaire_clean.groupby(
                question,
                dropna=False
            )[field]
            .apply(lambda x: x.isna().sum())
        )

# 16: Check duplicate records

In [ ]:
# DUPLICATE RECORD ANALYSIS

duplicate_report = []

for name, df in cleaned_data.items():

    duplicate_count = df.duplicated().sum()

    duplicate_report.append({
        "Dataset": name,
        "Total_Rows": len(df),
        "Duplicate_Rows": duplicate_count,
        "Duplicate_Percentage": round(
            (duplicate_count / len(df)) * 100, 2
        ) if len(df) > 0 else 0
    })


duplicate_report = pd.DataFrame(duplicate_report)

duplicate_report

# 17: Extract actual duplicate records

In [ ]:
# EXTRACT DUPLICATE RECORDS

duplicate_records = {}

for name, df in cleaned_data.items():

    duplicates = df[
        df.duplicated(keep=False)
    ].copy()

    duplicate_records[name] = duplicates

    if not duplicates.empty:

        print(f"\n{name}")
        print("Duplicate records:", len(duplicates))

        print(duplicates.head(5))

# 18: Save duplicate records and report

In [ ]:
# SAVE DUPLICATE ANALYSIS

duplicate_report.to_csv(
    os.path.join(TABLE_DIR, "duplicate_report.csv"),
    index=False
)

for name, df in duplicate_records.items():

    if not df.empty:

        df.to_csv(
            os.path.join(
                TABLE_DIR,
                f"{name}_duplicate_records.csv"
            ),
            index=False
        )

print("Duplicate analysis saved successfully.")

# 19: Identify categorical variables

In [ ]:
# CATEGORICAL DATA INSPECTION

categorical_report = []

for name, df in cleaned_data.items():

    for col in df.select_dtypes(
        include=["object", "string"]
    ).columns:

        unique_values = df[col].dropna().unique()

        categorical_report.append({
            "Dataset": name,
            "Column": col,
            "Unique_Categories": len(unique_values),
            "Categories": ", ".join(
                map(str, unique_values[:20])
            )
        })


categorical_report = pd.DataFrame(categorical_report)

categorical_report

# 20: Standardize Yes/No responses

In [ ]:
# STANDARDIZE YES/NO RESPONSES

yes_no_columns = [
    "Tillage",
    "Surface_cultivation",
    "Plough",
    "Harrow",
    "Rotavator",
    "Sand",
    "Loam",
    "Silt",
    "Clay",
    "Ploughing",
    "Burning",
    "Waste_collection",
    "Selling",
    "Littering",
    "Recycling",
    "Storage",
    "Reuse",
    "Coated_fertiliser_use",
    "Compost_use",
    "Cow_manure",
    "Chicken_manure",
    "Pig_manure",
    "Sheep_manure",
    "Goat_manure",
    "Microbial_Organic_fertiliser",
    "Commercial_compost",
    "Sewage_sludge_use",
    "Irrigation_pipes",
    "Nets",
    "Greenhouses",
    "Polytunnels",
    "Packaging",
    "Drums",
    "Barriers",
    "Crates"
]

for col in yes_no_columns:

    if col in questionnaire_clean.columns:

        questionnaire_clean[col] = (
            questionnaire_clean[col]
            .str.strip()
            .str.capitalize()
        )


print("Yes/No responses standardized.")

# 21 Check country and polymer categories

In [ ]:
# CHECK COUNTRY VALUES

country_datasets = [
    "mp_quant",
    "soil_props",
    "questionnaire",
    "mp_id",
    "std_quant",
    "std_polymer",
    "lab_meta"
]

for name in country_datasets:

    if "Country" in cleaned_data[name].columns:

        print(f"\n{name}")
        print(
            cleaned_data[name]["Country"]
            .value_counts(dropna=False)
        )


# CHECK POLYMER IDENTIFICATION VALUES

for name in ["mp_id", "std_polymer", "std_lib_uk"]:

    print(f"\n{name}: Polymer categories")

    print(
        cleaned_data[name]["ID"]
        .value_counts(dropna=False)
    )

# 22: Save categorical inspection report

In [ ]:
categorical_report.to_csv(
    os.path.join(TABLE_DIR, "categorical_quality_report.csv"),
    index=False
)

print("Categorical quality report saved.")

# 23: Numeric data quality report

In [ ]:
# NUMERIC DATA QUALITY ANALYSIS

numeric_report = []

for name, df in cleaned_data.items():

    for col in df.select_dtypes(include="number").columns:

        numeric_report.append({
            "Dataset": name,
            "Column": col,
            "Minimum": df[col].min(),
            "Maximum": df[col].max(),
            "Mean": round(df[col].mean(), 2),
            "Missing_Values": df[col].isna().sum(),
            "Negative_Values": (df[col] < 0).sum(),
            "Zero_Values": (df[col] == 0).sum()
        })


numeric_report = pd.DataFrame(numeric_report)

numeric_report

# 24: Check invalid numeric ranges

In [ ]:
# DEFINE NUMERIC VALIDATION RULES

validation_rules = {

    "mp_quant": {
        "MP_number": (0, None),
        "Depth": (10, 30)
    },

    "soil_props": {
        "pH": (0, 14),
        "EC": (0, None),
        "NH4": (0, None),
        "NO3": (0, None),
        "DOC": (0, None),
        "PO4": (0, None),
        "Depth": (10, 30)
    },

    "mp_id": {
        "Quality": (0, 100),
        "Depth": (10, 30)
    },

    "std_quant": {
        "Particle_number": (0, None)
    },

    "std_polymer": {
        "MatchQuality": (0, 100),
        "Particle": (1, None)
    },

    "std_lib_uk": {
        "MatchQuality": (0, 100),
        "Particle": (1, None)
    },

    "lab_meta": {
        "Nile_red_purity": (0, 100)
    }
}


# CHECK VALUES AGAINST RULES

invalid_numeric_records = []

for name, rules in validation_rules.items():

    df = cleaned_data[name]

    for col, (minimum, maximum) in rules.items():

        if col not in df.columns:
            continue

        invalid = pd.Series(False, index=df.index)

        if minimum is not None:
            invalid |= df[col] < minimum

        if maximum is not None:
            invalid |= df[col] > maximum

        for index in df[invalid].index:

            invalid_numeric_records.append({
                "Dataset": name,
                "Column": col,
                "Row_Index": index,
                "Value": df.loc[index, col],
                "Minimum_Allowed": minimum,
                "Maximum_Allowed": maximum
            })


invalid_numeric_records = pd.DataFrame(
    invalid_numeric_records
)

print("Invalid numeric observations:",
      len(invalid_numeric_records))

invalid_numeric_records

# 25: Save numeric quality reports

In [ ]:
# SAVE NUMERIC QUALITY REPORTS

numeric_report.to_csv(
    os.path.join(TABLE_DIR, "numeric_quality_report.csv"),
    index=False
)

invalid_numeric_records.to_csv(
    os.path.join(TABLE_DIR, "invalid_numeric_records.csv"),
    index=False
)

print("Numeric quality reports saved successfully.")

# 26: Detect potential outliers using IQR

In [ ]:
# IQR OUTLIER DETECTION

outlier_columns = {
    "mp_quant": ["MP_number"],
    "soil_props": ["pH", "EC", "NH4", "NO3", "DOC", "PO4"],
    "std_quant": ["Particle_number"]
}

outlier_records = {}

for name, columns in outlier_columns.items():

    df = cleaned_data[name]

    for col in columns:

        values = df[col].dropna()

        Q1 = values.quantile(0.25)
        Q3 = values.quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[
            (df[col] < lower_bound) |
            (df[col] > upper_bound)
        ].copy()

        outlier_records[(name, col)] = outliers

        print(f"\nDataset: {name}")
        print("Variable:", col)
        print("Lower bound:", round(lower_bound, 2))
        print("Upper bound:", round(upper_bound, 2))
        print("Potential outliers:", len(outliers))

# 27: Examine potential outlier observations

In [ ]:
# INSPECT POTENTIAL OUTLIERS

for (name, col), outliers in outlier_records.items():

    if not outliers.empty:

        print(f"\n{name} - {col}")
        print(outliers.head(10))

# 28: Visualize outliers using boxplots

In [ ]:
# BOXPLOT ANALYSIS
outlier_box_color = "#F4A261"   # warning orange
outlier_flier_color = "#D62828" # alert red for the outliers

for name, columns in outlier_columns.items():

    df = cleaned_data[name]

    for col in columns:

        plt.figure(figsize=(8, 4))

        sns.boxplot(
            x=df[col],
            color=outlier_box_color,
            width=0.4,
            flierprops={
                "marker": "o",
                "markerfacecolor": outlier_flier_color,
                "markeredgecolor": outlier_flier_color,
                "markersize": 5,
                "alpha": 0.7
            }
        )

        plt.title(f"{col} Distribution - {name}")
        plt.xlabel(col)

        plt.tight_layout()

        plt.show()

# 29: Validate main survey identifiers

In [ ]:
# MAIN SURVEY DATA VALIDATION

print("=" * 60)
print("MAIN SURVEY DATA VALIDATION")
print("=" * 60)

# DEFINE DATASETS AND MERGE KEYS

survey_datasets = {
    "MP Quantification": mp_quant_clean,
    "Soil Properties": soil_props_clean
}

merge_keys = ["Site", "Country", "Depth"]


# CHECK REQUIRED COLUMNS

for name, df in survey_datasets.items():

    missing_columns = [
        col for col in merge_keys
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{name} is missing columns: {missing_columns}"
        )


# CHECK DATA IDENTIFIERS AND DEPTH VALUES

for name, df in survey_datasets.items():

    print(f"\n{name}")

    print(
        "Missing identifiers:",
        df[merge_keys].isna().sum().to_dict()
    )

    print(
        "Duplicate site-depth records:",
        df.duplicated(subset=merge_keys).sum()
    )

    print(
        "Number of unique sites:",
        df["Site"].nunique()
    )

    print(
        "Depth categories:",
        sorted(df["Depth"].dropna().unique())
    )

    bad_depth = df[
        ~df["Depth"].isin([10, 20, 30])
        & df["Depth"].notna()
    ]

    print("Unexpected depth values:", len(bad_depth))


print("\nData validation completed.")

# 30: Check consistency between quantification and soil properties

In [ ]:
# CHECK SITE-DEPTH CONSISTENCY

quant_keys = set(
    map(
        tuple,
        mp_quant_clean[merge_keys].drop_duplicates().values
    )
)

soil_keys = set(
    map(
        tuple,
        soil_props_clean[merge_keys].drop_duplicates().values
    )
)

quant_only = quant_keys - soil_keys
soil_only = soil_keys - quant_keys

print("Keys only in MP Quantification:", len(quant_only))
print("Keys only in Soil Properties:", len(soil_only))

if len(quant_only) == 0 and len(soil_only) == 0:
    print("\nBoth datasets contain matching site-depth records.")
else:
    print("\nSome site-depth records do not match.")

# 31: Validate questionnaire site identifiers

In [ ]:
# CHECK QUESTIONNAIRE IDENTIFIERS

questionnaire_keys = ["Site", "Country"]

print("Questionnaire rows:", len(questionnaire_clean))

print(
    "Unique questionnaire sites:",
    questionnaire_clean["Site"].nunique()
)

print(
    "Duplicate questionnaire site records:",
    questionnaire_clean.duplicated(
        subset=questionnaire_keys
    ).sum()
)

print(
    "Missing questionnaire identifiers:",
    questionnaire_clean[questionnaire_keys]
    .isna().sum().to_dict()
)


# CHECK WHETHER QUESTIONNAIRE SITES MATCH QUANTIFICATION SITES

quant_sites = set(
    map(
        tuple,
        mp_quant_clean[questionnaire_keys].drop_duplicates().values
    )
)

questionnaire_sites = set(
    map(
        tuple,
        questionnaire_clean[questionnaire_keys].drop_duplicates().values
    )
)

print(
    "\nSites missing from questionnaire:",
    len(quant_sites - questionnaire_sites)
)

print(
    "Extra sites in questionnaire:",
    len(questionnaire_sites - quant_sites)
)

# 32: Merge the main survey datasets

In [ ]:
# MERGE MAIN SURVEY DATASETS

merge_keys = ["Site", "Country", "Depth"]

# STEP 1: MERGE MP QUANTIFICATION AND SOIL PROPERTIES

master_data = pd.merge(
    mp_quant_clean,
    soil_props_clean,
    on=merge_keys,
    how="left",
    validate="one_to_one"
)

print("After merging MP Quantification and Soil Properties:")
print("Rows:", master_data.shape[0])
print("Columns:", master_data.shape[1])


# STEP 2: ADD QUESTIONNAIRE INFORMATION

master_data = pd.merge(
    master_data,
    questionnaire_clean,
    on=["Site", "Country"],
    how="left",
    validate="many_to_one",
    indicator=True
)

# CHECK MERGE RESULTS

print("\nQuestionnaire Merge Results:")
print(master_data["_merge"].value_counts())

# REMOVE MERGE INDICATOR

master_data = master_data.drop(columns=["_merge"])


# FINAL MASTER DATASET SUMMARY

print("\nMASTER DATASET")
print("------------------------------")
print("Rows:", master_data.shape[0])
print("Columns:", master_data.shape[1])
print("Duplicate rows:", master_data.duplicated().sum())

print("\nMaster dataset created successfully.")

# 33: Inspect the master dataset

In [ ]:
# INSPECT MASTER DATASET

print("Dataset shape:", master_data.shape)

print("\nColumn names:")
print(master_data.columns.tolist())

print("\nFirst 5 rows:")
print(master_data.head())

print("\nData types:")
print(master_data.dtypes)

# 34: Check master dataset quality

In [ ]:
# MASTER DATASET QUALITY CHECK

print("=" * 60)
print("MASTER DATASET QUALITY CHECK")
print("=" * 60)

print("\nDataset shape:", master_data.shape)

print("\nDuplicate rows:", master_data.duplicated().sum())

print("\nMissing values:")
print(
    master_data.isna().sum()
    .sort_values(ascending=False)
)

print("\nMissing value percentage:")
missing_percentage = (
    master_data.isna().mean() * 100
).sort_values(ascending=False)

print(missing_percentage.round(2))

# 35: Save the master dataset

In [ ]:
# SAVE MASTER DATASET

master_data.to_csv(
    os.path.join(OUTPUT_DIR, "master_survey_dataset.csv"),
    index=False
)

print("Master survey dataset saved successfully.")

# 36: Descriptive statistics

In [ ]:
# EDA: DESCRIPTIVE STATISTICS

numeric_summary = master_data.describe().T

numeric_summary = numeric_summary.round(2)

numeric_summary

# 37: Microplastic distribution overview

In [ ]:
# EDA: MICROPLASTIC DISTRIBUTION

plt.figure(figsize=(10, 5))

sns.histplot(
    data=master_data,
    x="MP_number",
    bins=30,
    kde=True,
    color="#6A4C93",          # deep purple fill
    edgecolor="white",
    line_kws={"color": "#FF595E", "linewidth": 2}   # contrasting KDE line
)

plt.title("Distribution of Soil Microplastic Concentration")
plt.xlabel("Microplastic Number")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "microplastic_distribution.png"),
    bbox_inches="tight"
)

plt.show()

# LOG TRANSFORMATION

In [ ]:
# LOG TRANSFORMATION OF MP NUMBER

master_data["Log_MP_number"] = np.log1p(master_data["MP_number"])

print("Original skewness:", master_data["MP_number"].skew())
print("Log-transformed skewness:", master_data["Log_MP_number"].skew())

In [ ]:
# DISTRIBUTION OF LOG-TRANSFORMED MP NUMBER

plt.figure(figsize=(10, 5))

sns.histplot(
    data=master_data,
    x="Log_MP_number",
    bins=30,
    kde=True,
    color="#457B9D"
)

plt.title("Distribution of Log-Transformed MP Number")
plt.xlabel("Log(1 + MP Number)")
plt.ylabel("Count")

plt.show()

# 38: Average microplastic concentration by country

In [ ]:
# EDA: MICROPLASTIC CONCENTRATION BY COUNTRY

country_summary = (
    master_data.groupby("Country")["MP_number"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
    .sort_values(by="mean", ascending=False)
)

country_summary

# 39: Visualize microplastic concentration by country

In [ ]:
# VISUALIZATION: MICROPLASTIC CONCENTRATION BY COUNTRY

plt.figure(figsize=(10, 5))

sns.barplot(
    data=master_data,
    x="Country",
    y="MP_number",
    errorbar=None,
    palette="viridis"
)

plt.title("Average Soil Microplastic Concentration by Country")
plt.xlabel("Country")
plt.ylabel("Average Microplastic Number")
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "microplastic_by_country.png"),
    bbox_inches="tight"
)

plt.show()

# 40: Microplastic concentration by soil depth

In [ ]:
# EDA: MICROPLASTIC CONCENTRATION BY DEPTH

depth_summary = (
    master_data.groupby("Depth")["MP_number"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

depth_summary

# 41: Visualize microplastic concentration by depth

In [ ]:
# VISUALIZATION: MICROPLASTIC CONCENTRATION BY DEPTH

depth_palette = {
    10: "#A8DADC",   # light blue
    20: "#457B9D",   # medium blue
    30: "#1D3557",   # dark navy
}

plt.figure(figsize=(9, 5))

sns.boxplot(
    data=master_data,
    x="Depth",
    y="MP_number",
    hue="Depth",
    palette=depth_palette,
    legend=False,
    width=0.5
)

plt.title("Microplastic Concentration Across Soil Depths")
plt.xlabel("Soil Depth (cm)")
plt.ylabel("Microplastic Number")

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "microplastic_by_depth.png"),
    bbox_inches="tight"
)

plt.show()

# 42: Correlation analysis

In [ ]:
# EDA: CORRELATION WITH SOIL PROPERTIES

soil_columns = [
    "pH", "EC", "NH4", "NO3", "DOC", "PO4"
]

correlation_data = master_data[
    ["MP_number"] + soil_columns
]

correlation_matrix = correlation_data.corr(
    method="pearson"
)

spearman_corr = master_data[
    ["MP_number", "pH", "EC", "NH4", "NO3", "DOC", "PO4"]
].corr(method="spearman")

print(spearman_corr)

mp_correlations = (
    correlation_matrix["MP_number"]
    .drop("MP_number")
    .sort_values(ascending=False)
)

print("Correlation with Microplastic Number:")
print(mp_correlations.round(3))

# 43: Correlation heatmap

In [ ]:
# VISUALIZATION: CORRELATION HEATMAP

plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Between Microplastics and Soil Properties")

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "soil_property_correlation.png"),
    bbox_inches="tight"
)

plt.show()

# 44: Scatter plots

In [ ]:
# VISUALIZATION: MICROPLASTICS VS SOIL PROPERTIES

scatter_colors = {
    "pH":  "#E63946",   # red
    "EC":  "#457B9D",   # steel blue
    "NH4": "#2A9D8F",   # teal
    "NO3": "#E9C46A",   # mustard
    "DOC": "#9B5DE5",   # purple
    "PO4": "#F4A261",   # orange
}

for col in soil_columns:

    plt.figure(figsize=(7, 5))

    sns.scatterplot(
        data=master_data,
        x=col,
        y="MP_number",
        color=scatter_colors.get(col, "#333333"),
        alpha=0.5,
        s=25,
        edgecolor="white",
        linewidth=0.3
    )

    plt.title(f"Microplastic Number vs {col}")
    plt.xlabel(col)
    plt.ylabel("Microplastic Number")

    plt.tight_layout()

    plt.savefig(
        os.path.join(FIG_DIR, f"mp_vs_{col}.png"),
        bbox_inches="tight"
    )

    plt.show()

# 45: Inspect agricultural practice categories

In [ ]:
# EDA: AGRICULTURAL PRACTICE CATEGORIES

practice_columns = [
    "Tillage",
    "Surface_cultivation",
    "Plough",
    "Harrow",
    "Rotavator",
    "PMF_type",
    "Coated_fertiliser_use",
    "Compost_use",
    "Sewage_sludge_use",
    "Irrigation_pipes",
    "Nets",
    "Greenhouses",
    "Polytunnels",
    "Packaging",
    "Drums",
    "Barriers",
    "Crates"
]

for col in practice_columns:

    if col in questionnaire_clean.columns:

        print("\n" + "=" * 45)
        print("Column:", col)
        print(questionnaire_clean[col].value_counts(dropna=False))

# 46: Visualize selected agricultural practices

In [ ]:
# VISUALIZATION: AGRICULTURAL PLASTIC MATERIALS

plastic_columns = [
    "Irrigation_pipes",
    "Nets",
    "Greenhouses",
    "Polytunnels",
    "Packaging",
    "Drums",
    "Barriers",
    "Crates"
]

plastic_usage = []

for col in plastic_columns:

    if col in questionnaire_clean.columns:

        counts = questionnaire_clean[col].value_counts()

        for category, count in counts.items():

            plastic_usage.append({
                "Material": col,
                "Category": category,
                "Count": count
            })

plastic_usage = pd.DataFrame(plastic_usage)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=plastic_usage,
    x="Material",
    y="Count",
    hue="Category",
    palette="Set2"
)

plt.title("Reported Use of Agricultural Plastic Materials")
plt.xlabel("Agricultural Plastic Material")
plt.ylabel("Number of Sites")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "agricultural_plastic_usage.png"),
    bbox_inches="tight"
)

plt.show()

# 47: Compare microplastic concentration by agricultural practices

In [ ]:
# AGRICULTURAL PRACTICES VS MICROPLASTIC CONCENTRATION

practice_analysis_columns = [
    "Irrigation_pipes",
    "Nets",
    "Greenhouses",
    "Polytunnels",
    "Packaging",
    "Compost_use",
    "Coated_fertiliser_use"
]

practice_results = []

for col in practice_analysis_columns:

    if col in master_data.columns:

        summary = (
            master_data.groupby(col)["MP_number"]
            .agg(["count", "mean", "median"])
            .reset_index()
        )

        summary["Practice"] = col

        summary = summary.rename(
            columns={col: "Category"}
        )

        practice_results.append(summary)


practice_comparison = pd.concat(
    practice_results,
    ignore_index=True
)

practice_comparison = practice_comparison[
    ["Practice", "Category", "count", "mean", "median"]
]

practice_comparison.round(2)

# 48: Visualize agricultural practice comparisons

In [ ]:
# VISUALIZATION: AGRICULTURAL PRACTICES VS MICROPLASTICS

for col in practice_analysis_columns:

    if col in master_data.columns:

        plt.figure(figsize=(8, 5))

        sns.barplot(
            data=master_data,
            x=col,
            y="MP_number",
            errorbar=None,
            hue=col,
            palette="coolwarm",
            legend=False
        )

        plt.title(f"Microplastic Concentration by {col}")
        plt.xlabel(col)
        plt.ylabel("Average Microplastic Number")
        plt.xticks(rotation=30)

        plt.tight_layout()

        plt.savefig(
            os.path.join(FIG_DIR, f"mp_by_{col}.png"),
            bbox_inches="tight"
        )

        plt.show()

# 49: Analyze identified polymer types
# EDA: Polymer identification

In [ ]:
# EDA: POLYMER IDENTIFICATION FREQUENCY

polymer_summary = (
    mp_id_clean["ID"]
    .value_counts()
    .reset_index()
)

polymer_summary.columns = [
    "Polymer",
    "Frequency"
]

polymer_summary["Percentage"] = (
    polymer_summary["Frequency"]
    / polymer_summary["Frequency"].sum()
    * 100
).round(2)

polymer_summary

# 50: Visualize polymer identification

In [ ]:
# VISUALIZATION: POLYMER IDENTIFICATION

from matplotlib.colors import LinearSegmentedColormap

top_polymers = polymer_summary.head(15).copy()

# Custom strawberry -> matcha gradient
strawberry_matcha = LinearSegmentedColormap.from_list(
    "strawberry_matcha",
    ["#E8748C", "#F2A8B8", "#F5E6D3", "#C8D8A8", "#A8C686"]
)

polymer_colors = [
    strawberry_matcha(x)
    for x in np.linspace(0, 1, len(top_polymers))
]

plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_polymers,
    x="Frequency",
    y="Polymer",
    hue="Polymer",
    palette=polymer_colors,
    legend=False
)

# Simple percentage labels
for i, row in enumerate(top_polymers.itertuples()):
    plt.text(
        row.Frequency + 30,
        i,
        f"{row.Percentage:.1f}%",
        va="center",
        fontsize=9,
        color="#555555"
    )

plt.title("Top 15 Identified Polymer Types", fontsize=13)
plt.xlabel("Identification Frequency")
plt.ylabel("")

sns.despine()

plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, "polymer_identification.png"),
    bbox_inches="tight"
)
plt.show()

# 51: Generate the analysis summary

In [ ]:
# FINAL EDA SUMMARY

eda_summary = pd.DataFrame({
    "Analysis": [
        "Total survey observations",
        "Number of countries",
        "Number of unique sites",
        "Average microplastic number",
        "Median microplastic number",
        "Minimum microplastic number",
        "Maximum microplastic number",
        "Number of identified polymer types"
    ],

    "Result": [
        len(master_data),
        master_data["Country"].nunique(),
        master_data["Site"].nunique(),
        round(master_data["MP_number"].mean(), 2),
        round(master_data["MP_number"].median(), 2),
        master_data["MP_number"].min(),
        master_data["MP_number"].max(),
        mp_id_clean["ID"].nunique()
    ]
})

eda_summary

# 52: Save final analysis outputs

In [ ]:
# SAVE FINAL ANALYSIS OUTPUTS

eda_summary.to_csv(
    os.path.join(TABLE_DIR, "final_eda_summary.csv"),
    index=False
)

country_summary.to_csv(
    os.path.join(TABLE_DIR, "country_comparison.csv")
)

depth_summary.to_csv(
    os.path.join(TABLE_DIR, "depth_comparison.csv")
)

mp_correlations.to_csv(
    os.path.join(TABLE_DIR, "soil_property_correlations.csv")
)

practice_comparison.to_csv(
    os.path.join(TABLE_DIR, "agricultural_practice_comparison.csv"),
    index=False
)

polymer_summary.to_csv(
    os.path.join(TABLE_DIR, "polymer_identification_summary.csv"),
    index=False
)

master_data.to_csv(
    os.path.join(OUTPUT_DIR, "master_survey_dataset.csv"),
    index=False
)

print("Final analysis outputs saved successfully.")

# 53: Inspect duplicate polymer records

In [ ]:
# INVESTIGATE DUPLICATE POLYMER IDENTIFICATION RECORDS

mp_id_duplicates = mp_id_clean[
    mp_id_clean.duplicated(keep=False)
].copy()

print("Total duplicate records:", len(mp_id_duplicates))

print("\nDuplicate records by country:")
print(
    mp_id_duplicates["Country"]
    .value_counts()
)

print("\nSample duplicate records:")
mp_id_duplicates.head(10)

# 54: Check whether duplicates affect polymer frequencies

In [ ]:
# COMPARE POLYMER FREQUENCIES BEFORE AND AFTER DUPLICATE REMOVAL

mp_id_unique = mp_id_clean.drop_duplicates().copy()

polymer_frequency_comparison = pd.DataFrame({
    "Original_Frequency": mp_id_clean["ID"].value_counts(),
    "After_Duplicate_Removal": mp_id_unique["ID"].value_counts()
}).fillna(0).astype(int)

polymer_frequency_comparison["Difference"] = (
    polymer_frequency_comparison["Original_Frequency"]
    - polymer_frequency_comparison["After_Duplicate_Removal"]
)

polymer_frequency_comparison = (
    polymer_frequency_comparison
    .sort_values("Original_Frequency", ascending=False)
)

polymer_frequency_comparison

# 55: Save final data-quality investigation

In [ ]:
# SAVE FINAL DATA QUALITY INVESTIGATION

mp_id_duplicates.to_csv(
    os.path.join(
        TABLE_DIR,
        "polymer_identification_duplicates.csv"
    ),
    index=False
)

polymer_frequency_comparison.to_csv(
    os.path.join(
        TABLE_DIR,
        "polymer_frequency_comparison.csv"
    )
)

# SAVE FINAL MASTER DATASET

master_data.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "master_survey_dataset.csv"
    ),
    index=False
)

print("Final data-quality investigation saved.")
print("Master survey dataset saved.")
print("Main analysis outputs are ready.")